# SonicSight-DINOv2: Audio-Visual Source Separation

Kaggle submission notebook for the audio-visual source separation task.

**Model**: DINOv2-guided Cross-Modal Attention + Audio U-Net
**Phases**: 1. Audio-only, 2. Cross-modal warmup, 3. End-to-end fine-tuning

In [ ]:
!pip install -q pytorch-lightning torchaudio torchmetrics mir_eval jiwer einops transformers timm

In [ ]:
# Setup paths
import sys
sys.path.insert(0, '/kaggle/working/SonicSightDino')

import os
os.chdir('/kaggle/working/SonicSightDino')

# Verify structure
!ls -la
!ls -la /kaggle/input/

In [ ]:
# Preprocess data if needed
from scripts.preprocess_data import main as preprocess_main
import argparse

# Check if cache exists
if not os.path.exists('/kaggle/input/cache/index.json'):
    print("Running preprocessing...")
    !python scripts/preprocess_data.py --input_dir /kaggle/input --output_dir /kaggle/working/cache --n_sources 4
else:
    print("Cache exists, skipping preprocessing")

In [ ]:
# Train Phase 1: Audio-only
!python scripts/train.py \
    --config configs/kaggle.yaml \
    --phase phase1 \
    --max_steps 10000 \
    --checkpoint_dir /kaggle/working/checkpoints/phase1

In [ ]:
# Train Phase 2: Cross-modal attention warmup
!python scripts/train.py \
    --config configs/kaggle.yaml \
    --phase phase2 \
    --max_steps 10000 \
    --checkpoint_dir /kaggle/working/checkpoints/phase2 \
    --resume_from_checkpoint /kaggle/working/checkpoints/phase1/last.ckpt

In [ ]:
# Train Phase 3: End-to-end fine-tuning with progressive difficulty
!python scripts/train.py \
    --config configs/kaggle.yaml \
    --phase phase3 \
    --max_steps 40000 \
    --checkpoint_dir /kaggle/working/checkpoints/phase3 \
    --resume_from_checkpoint /kaggle/working/checkpoints/phase2/last.ckpt

In [ ]:
# Run evaluation
!python scripts/run_evaluation.py \
    --checkpoint /kaggle/working/checkpoints/phase3/best.ckpt \
    --index_file /kaggle/input/cache/index.json \
    --output /kaggle/working/eval_results.json \
    --n_sources 4

In [ ]:
# Generate Kaggle submission
!python scripts/kaggle_submission.py \
    --checkpoint /kaggle/working/checkpoints/phase3/best.ckpt \
    --index_file /kaggle/input/cache/index.json \
    --output /kaggle/working/submission.csv \
    --n_sources 4